# Motor Current Monitor

Reads `Present_Load` and `Present_Position` from the `shoulder_pan` servo (Feetech STS3215 bus) every 100 ms and displays them as a live plot.

The motor sweeps incrementally between ±30°, stepping 1° every 50 ms and toggling direction based on the starting position.

**Goal:** Investigate `Present_Current` from the servo. Compare it against `Present_Load` to understand how each quantity responds to increased motor torque.

In [13]:
_already_imported = "SO101Follower" in dir()

if not _already_imported:
    from lerobot.robots.so_follower import SO101FollowerConfig, SO101Follower
    from dash import Dash, dcc, html, Input, Output, callback
    from dash._jupyter import _dash_comm
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    from werkzeug.serving import make_server
    import collections
    import logging
    import threading
    import time

if _already_imported:
    print("Already imported — kernel was already running.")
else:
    print("Imported fresh — kernel was recently restarted.")

Already imported — kernel was already running.

In [14]:
PORT = "/dev/serial/by-id/usb-1a86_USB_Single_Serial_5AAF263835-if00"
ROBOT_ID = "my_so101_follower"

In [15]:
config = SO101FollowerConfig(port=PORT, id=ROBOT_ID)
robot = SO101Follower(config)
robot.connect(calibrate=False)
print(f"Connected on {PORT}")

Connected on /dev/serial/by-id/usb-1a86_USB_Single_Serial_5AAF263835-if00

In [16]:
print("Robot connected:", robot.is_connected)

Robot connected: True

In [17]:
_N = 200
_load_buf = collections.deque([0.0] * _N, maxlen=_N)
_pos_buf = collections.deque([0.0] * _N, maxlen=_N)
_current_buf = collections.deque([0.0] * _N, maxlen=_N)
_temp_buf = collections.deque([0.0] * _N, maxlen=_N)

_HOST = "127.0.0.1"
_DASH_PORT = 8060
_URL = f"http://{_HOST}:{_DASH_PORT}"

app = Dash(__name__, update_title=None)
app.layout = html.Div([
    dcc.Graph(id="motor-graph", style={"height": "675px"}),
    dcc.Interval(id="interval", interval=100, n_intervals=0),
])

@callback(
    Output("motor-graph", "figure"),
    Input("interval", "n_intervals"),
)
def update(n):
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05)
    fig.add_trace(
        go.Scatter(y=list(_load_buf), name="Load", line=dict(color="steelblue")),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(y=list(_current_buf), name="Current", line=dict(color="mediumseagreen")),
        row=2, col=1,
    )
    fig.add_trace(
        go.Scatter(y=list(_pos_buf), name="Position (°)", line=dict(color="darkorange")),
        row=3, col=1,
    )
    fig.add_trace(
        go.Scatter(y=list(_temp_buf), name="Temperature (°C)", line=dict(color="tomato")),
        row=4, col=1,
    )
    fig.update_yaxes(title_text="Present_Load", row=1, col=1)
    fig.update_yaxes(title_text="Present_Current", row=2, col=1)
    fig.update_yaxes(title_text="Position (°)", row=3, col=1)
    fig.update_yaxes(title_text="Temperature (°C)", row=4, col=1)
    fig.update_layout(margin=dict(l=60, r=20, t=20, b=40), height=675)
    return fig

logging.getLogger("werkzeug").setLevel(logging.ERROR)

_dash_server = make_server(_HOST, _DASH_PORT, app.server, threaded=True)
threading.Thread(target=_dash_server.serve_forever, daemon=True).start()
_dash_comm.send({"type": "show", "port": _DASH_PORT, "url": _URL})

In [18]:
obs = robot.get_observation()
target = obs["shoulder_pan.pos"]
direction = -1.0 if target > 0 else 1.0
pause_until = 0.0

try:
    while True:
        obs = robot.get_observation()
        pos = obs["shoulder_pan.pos"]
        load = robot.bus.read("Present_Load", "shoulder_pan")
        current = robot.bus.read("Present_Current", "shoulder_pan")
        temp = robot.bus.read("Present_Temperature", "shoulder_pan")

        _load_buf.append(load)
        _pos_buf.append(pos)
        _current_buf.append(current)
        _temp_buf.append(temp)

        now = time.perf_counter()

        if target >= 30.0 or target <= -30.0:
            direction *= -1.0
            target += direction
            target = max(-30.0, min(30.0, target))
            robot.send_action({"shoulder_pan.pos": target})
            pause_until = now + 1.0
        elif now >= pause_until:
            target += direction
            target = max(-30.0, min(30.0, target))
            robot.send_action({"shoulder_pan.pos": target})

        time.sleep(0.05)

except KeyboardInterrupt:
    pass

print("Loop stopped.")

Loop stopped.

In [19]:
_dash_server.shutdown()
robot.disconnect()
print("Disconnected.")

Disconnected.